# E-learning Platform — Data Generator
### Portfolio Project: Who Learns Online? Data-Driven Segmentation as a Foundation for Product Decisions

RFM Segmentation + Behavioral Analysis

The script generates synthetic data for an e-learning platform using the **Faker** library.
The output consists of 5 CSV files ready for RFM analysis in SQL and Power BI.

| Table | Description |
|---|---|
| `customers.csv` | Customers with joining date |
| `courses.csv` | Catalogue of 15 courses with categories and prices |
| `purchases.csv` | Course purchases made by customers |
| `assessments.csv` | Progress, completion rate and course ratings |
| `logins.csv` | Login sessions (date + duration) |


## 1. Installation and importation

In [ ]:
# Installation and importation
!pip install faker -q

import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
from google.colab import files   # to download generated CSV files



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.7 MB/s eta 0:00:00


## 2. Configuration – seeds and global parameters

In [ ]:
# The random seed is used to initialize the random number generator
Faker.seed(60)
np.random.seed(60)
random.seed(60)
fake = Faker("en_US")

# Setting global parameters: number of Customers and time gap
N_CUSTOMERS = 1000
START_DATE  = datetime(2022, 1, 1)
END_DATE    = datetime(2024, 12, 31)



## 3. Pre-defined table of Courses

In [ ]:
courses = pd.DataFrame([
    ("C001", "Adobe Illustrator - Master Vector Illustration",        "Software & Tools",        99),
    ("C002", "Adobe Photoshop - Make Images Striking",                "Software & Tools",        89),
    ("C003", "Adobe InDesign - Layouts & Editorial Design",           "Software & Tools",        79),
    ("C004", "Web Design in Figma - Craft User Experience",           "Software & Tools",        99),
    ("C005", "Canva - Explore Interactive Presentation",              "Software & Tools",        49),
    ("C006", "Animation from Scratch - Put Illustration in Motion",   "Design Theory",           49),
    ("C007", "Color Theory - Guide for Graphic Design",               "Design Theory",           19),
    ("C008", "Moodboard Creation - Boost your Creativity",            "Design Theory",           29),
    ("C009", "Book Cover Design - Visualise Content Effectively",     "Design Theory",           39),
    ("C010", "Typography Design - Visual Dimension of Language ",     "Design Theory",           49),
    ("C011", "Logo Design - Create Iconic Symbols",                   "Brand & Communication",   79),
    ("C012", "Presentation Design - Attract & Impact",                "Brand & Communication",   49),
    ("C013", "Infographic Design - Make Data Visual",                 "Brand & Communication",   69),
    ("C014", "Visual Communication - Storytelling Techniques",        "Brand & Communication",   79),
    ("C015", "Visual Branding - From Concept to Strategy",            "Brand & Communication",   89),
], columns=["course_id", "course_name", "course_category", "course_price"])

course_ids = courses["course_id"].tolist()
courses


,course_id,course_name,course_category,course_price
0,C001,Adobe Illustrator - Master Vector Illustration,Software & Tools,99
1,C002,Adobe Photoshop - Make Images Striking,Software & Tools,89
2,C003,Adobe InDesign - Layouts & Editorial Design,Software & Tools,79
3,C004,Web Design in Figma - Craft User Experience,Software & Tools,99
4,C005,Canva - Explore Interactive Presentation,Software & Tools,49
5,C006,Animation from Scratch - Put Illustration in M...,Design Theory,49
6,C007,Color Theory - Guide for Graphic Design,Design Theory,19
7,C008,Moodboard Creation - Boost your Creativity,Design Theory,29
8,C009,Book Cover Design - Visualise Content Effectively,Design Theory,39
9,C010,Typography Design - Visual Dimension of Language,Design Theory,49


## 4. Behavioral profiles

Each customer is randomly assigned to one of 6 profiles.
The profiles reflect real behavioral patterns observed on e-learning platforms
and will correspond to RFM segments identified in the later analysis.

| Profile | Behavioral description |
|---|---|
| **champion** | Buys often, completes courses, gives high ratings |
| **loyal** | Regular user, returns for new courses |
| **collector** | Buys a lot, rarely finishes — high M, low completion |
| **at_risk** | Was active, recently disappeared (120–365 days ago) |
| **hibernating** | Long inactive, 1–2 purchases in history |
| **new** | Joined recently, in the exploration phase |


In [ ]:
# recency_days — how many days before END_DATE the last purchase took place (stands for R in RFM)
# completion_prob — the probability of finishing a course
# n_sessions — number of logins per one course
# session_dur — duration of a session [minutes]

PROFILES = {
    "champion": {
        "weight":          0.20,
        "n_purchases":     (5, 12),
        "recency_days":    (1, 60),
        "completion_prob": 0.85,
        "rating_mean":     4.5,  "rating_std": 0.4,
        "n_sessions":      (10, 20),
        "session_dur":     (30, 150),
    },
    "loyal": {
        "weight":          0.25,
        "n_purchases":     (3, 7),
        "recency_days":    (1, 120),
        "completion_prob": 0.70,
        "rating_mean":     4.0,  "rating_std": 0.5,
        "n_sessions":      (8, 16),
        "session_dur":     (20, 120),
    },
    "collector": {
        "weight":          0.20,
        "n_purchases":     (4, 10),
        "recency_days":    (30, 180),
        "completion_prob": 0.15,
        "rating_mean":     3.5,  "rating_std": 0.7,
        "n_sessions":      (1, 5),
        "session_dur":     (5, 30),
    },
    "at_risk": {
        "weight":          0.15,
        "n_purchases":     (2, 5),
        "recency_days":    (120, 365),
        "completion_prob": 0.55,
        "rating_mean":     3.8,  "rating_std": 0.6,
        "n_sessions":      (4, 12),
        "session_dur":     (15, 60),
    },
    "hibernating": {
        "weight":          0.10,
        "n_purchases":     (1, 2),
        "recency_days":    (365, 730),
        "completion_prob": 0.35,
        "rating_mean":     3.2,  "rating_std": 0.8,
        "n_sessions":      (1, 10),
        "session_dur":     (5, 25),
    },
    "new": {
        "weight":          0.10,
        "n_purchases":     (1, 3),
        "recency_days":    (1, 30),
        "completion_prob": 0.40,
        "rating_mean":     4.0,  "rating_std": 0.5,
        "n_sessions":      (1, 10),
        "session_dur":     (10, 60),
    },
}



## 5. Helper functions

In [ ]:
def clamp(value, lo, hi):
    """Clamps a value to the range [lo, hi]."""
    return max(lo, min(hi, value))

def rand_date_between(start: datetime, end: datetime) -> datetime:
    """Returns a random date within the range [start, end]."""
    delta = (end - start).days
    if delta <= 0:
        return start
    return start + timedelta(days=random.randint(0, delta))


## 6. Data generation

Main loop — for each customer:
1. Assigns a random behavioral profile
2. Calculates the joining date and purchase dates
3. Generates login sessions consistent with the profile
4. Creates an assessment only if the customer had at least 1 session


In [ ]:
profile_names   = list(PROFILES.keys())
profile_weights = [PROFILES[p]["weight"] for p in profile_names]

customers_rows, purchases_rows = [], []
assessments_rows, logins_rows  = [], []
t_id, a_id, l_id = 1, 1, 1

# Course quality modifiers — applied on top of profile-level rating and completion rate.
# Positive values indicate a well-received, engaging course; negative values reflect
# poor reception or high difficulty. Kept in range ±0.3 to avoid clipping at boundaries.
COURSE_QUALITY = {
    "C001": {"rating_bonus": +0.25, "completion_bonus": +0.10},
    "C002": {"rating_bonus": -0.10, "completion_bonus": -0.05},
    "C003": {"rating_bonus": +0.20, "completion_bonus": +0.08},
    "C004": {"rating_bonus": +0.15, "completion_bonus": +0.05},
    "C005": {"rating_bonus": -0.20, "completion_bonus": -0.10},
    "C006": {"rating_bonus": +0.30, "completion_bonus": +0.12},
    "C007": {"rating_bonus": -0.25, "completion_bonus": +0.08},
    "C008": {"rating_bonus": -0.15, "completion_bonus": -0.12},
    "C009": {"rating_bonus": +0.20, "completion_bonus": +0.10},
    "C010": {"rating_bonus": -0.10, "completion_bonus": +0.06},
    "C011": {"rating_bonus": +0.10, "completion_bonus": +0.15},
    "C012": {"rating_bonus": +0.25, "completion_bonus": +0.08},
    "C013": {"rating_bonus": +0.30, "completion_bonus": +0.20},
    "C014": {"rating_bonus": -0.30, "completion_bonus": -0.15},
    "C015": {"rating_bonus": +0.05, "completion_bonus": +0.03},
}

for cust_idx in range(N_CUSTOMERS):
    customer_id  = f"U{str(cust_idx + 1).zfill(4)}"
    profile_name = random.choices(profile_names, weights=profile_weights, k=1)[0]

    p = PROFILES[profile_name]

    # Date of the last purchase (recency counted backwards from END_DATE)
    recency       = random.randint(*p["recency_days"])
    last_purchase = END_DATE - timedelta(days=recency)
    last_purchase = max(last_purchase, START_DATE + timedelta(days=30))

    n_purchases = min(random.randint(*p["n_purchases"]), len(course_ids))

    # joining_date is always before the first purchase
    joining_date = rand_date_between(
        START_DATE,
        last_purchase - timedelta(days=n_purchases)
    )

    customers_rows.append({
        "customer_id":  customer_id,
        "joining_date": joining_date.strftime("%Y-%m-%d"),
    })

    # Courses without repetition
    chosen_courses = random.sample(course_ids, n_purchases)

    # Dates of purchases distributed between joining_date and last_purchase
    if n_purchases == 1:
        purchase_dates = [last_purchase]
    else:
        mid_dates = sorted([
            rand_date_between(joining_date + timedelta(days=1), last_purchase)
            for _ in range(n_purchases - 1)
        ])
        purchase_dates = mid_dates + [last_purchase]

    for i, course_id in enumerate(chosen_courses):
        purchase_date = purchase_dates[i]
        course_price  = int(courses.loc[
            courses["course_id"] == course_id, "course_price"
        ].values[0])
        amount_paid = course_price

        purchases_rows.append({
            "purchase_id":  f"P{str(t_id).zfill(6)}",
            "customer_id":  customer_id,
            "course_id":    course_id,
            "purchase_date": purchase_date.strftime("%Y-%m-%d"),
            "amount_paid":  amount_paid,
        })
        t_id += 1

        # LOGINS
        n_sessions = random.randint(*p["n_sessions"])
        login_end  = min(END_DATE, purchase_date + timedelta(days=180))

        session_dates = sorted([
            rand_date_between(purchase_date, login_end)
            for _ in range(n_sessions)
        ])

        for sess_date in session_dates:
            logins_rows.append({
                "login_id":       f"L{str(l_id).zfill(7)}",
                "customer_id":    customer_id,
                "course_id":      course_id,
                "login_date":     sess_date.strftime("%Y-%m-%d"),
                "login_duration": random.randint(*p["session_dur"]),
            })
            l_id += 1

        # ASSESSMENT
        # Generated only if the customer had at least 1 session
        if n_sessions >= 1:
            quality      = COURSE_QUALITY[course_id]
            did_complete = random.random() < p["completion_prob"]

            if did_complete:
                completion_rate = clamp(
                    round(random.uniform(0.80, 1.0) + quality["completion_bonus"], 2),
                    0.0, 1.0
                )
                completion_date = rand_date_between(
                    purchase_date + timedelta(days=1), login_end
                ).strftime("%Y-%m-%d")
            else:
                completion_rate = clamp(
                    round(random.uniform(0.02, 0.79) + quality["completion_bonus"], 2),
                    0.0, 1.0
                )
                completion_date = None

            # 70% of users with at least 1 session provide a rating
            if random.random() < 0.70:
                # Rating rounded to 0.5 (same as Udemy, Coursera)
                raw_rating = np.random.normal(p["rating_mean"], p["rating_std"])
                raw_rating += quality["rating_bonus"]
                rating = clamp(round(raw_rating * 2) / 2, 0.5, 5.0)
            else:
                rating = None

            assessments_rows.append({
                "assessment_id":   f"A{str(a_id).zfill(6)}",
                "customer_id":     customer_id,
                "course_id":       course_id,
                "completion_rate": completion_rate,
                "rating":          rating,
                "completion_date": completion_date,
            })
            a_id += 1


## 7. Creating DataFrames


In [ ]:
customers_df   = pd.DataFrame(customers_rows)
purchases_df   = pd.DataFrame(purchases_rows)
assessments_df = pd.DataFrame(assessments_rows)
logins_df      = pd.DataFrame(logins_rows)

print("Table sizes")
print(f"  customers:    {len(customers_df):>6} rows")
print(f"  purchases:    {len(purchases_df):>6} rows")
print(f"  assessments:  {len(assessments_df):>6} rows")
print(f"  logins:       {len(logins_df):>6} rows")
print(f"  courses:      {len(courses):>6} rows")


Table sizes
  customers:      1000 rows
  purchases:      5217 rows
  assessments:    5217 rows
  logins:        51609 rows
  courses:          15 rows


## 8. Data Preview

In [ ]:
customers_df.head()

,customer_id,joining_date
0,U0001,2022-09-29
1,U0002,2022-06-12
2,U0003,2023-08-03
3,U0004,2024-10-05
4,U0005,2022-01-02


In [ ]:
purchases_df.head()

,purchase_id,customer_id,course_id,purchase_date,amount_paid
0,P000001,U0001,C004,2022-11-11,99
1,P000002,U0001,C013,2023-01-02,69
2,P000003,U0001,C008,2023-09-05,29
3,P000004,U0001,C014,2024-10-18,79
4,P000005,U0002,C014,2023-01-08,79


In [ ]:
assessments_df.head()

,assessment_id,customer_id,course_id,completion_rate,rating,completion_date
0,A000001,U0001,C004,1.00,3.5,2022-12-20
1,A000002,U0001,C013,0.91,3.5,2023-01-26
2,A000003,U0001,C008,0.81,NaN,2024-01-07
3,A000004,U0001,C014,0.91,NaN,2024-11-09
4,A000005,U0002,C014,0.80,4.5,2023-02-24


In [ ]:
logins_df.head()

,login_id,customer_id,course_id,login_date,login_duration
0,L0000001,U0001,C004,2022-11-13,99
1,L0000002,U0001,C004,2022-11-22,100
2,L0000003,U0001,C004,2022-12-14,106
3,L0000004,U0001,C004,2022-12-20,39
4,L0000005,U0001,C004,2022-12-23,85


## 9. Descriptive statistics

In [ ]:
print("completion_rate")
print(assessments_df["completion_rate"].describe().round(2))
print()
print("rating (rated courses only)")
print(assessments_df["rating"].dropna().describe().round(2))
print()
print("Completed vs incomplete courses")
print(assessments_df["completion_date"]
      .isna()
      .map({True: "Incomplete", False: "Completed"})
      .value_counts()
      .to_string())


completion_rate
count    5254.00
mean        0.69
std         0.29
min         0.02
25%         0.47
50%         0.82
75%         0.91
max         1.00
Name: completion_rate, dtype: float64

rating (rated courses only)
count    3697.00
mean        3.99
std         0.70
min         1.00
25%         3.50
50%         4.00
75%         4.50
max         5.00
Name: rating, dtype: float64

Completed vs incomplete courses
completion_date
Completed     2952
Incomplete    2302


## 10. Save and download CSV files

Files are saved in the Colab environment and then automatically downloaded.


In [ ]:
courses.to_csv(        "courses.csv",      index=False)
customers_df.to_csv(   "customers.csv",    index=False)
purchases_df.to_csv(   "purchases.csv",    index=False)
assessments_df.to_csv( "assessments.csv",  index=False)
logins_df.to_csv(      "logins.csv",       index=False)

for fname in ["courses.csv", "customers.csv", "purchases.csv",
              "assessments.csv", "logins.csv"]:
    files.download(fname)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>